### Recuit Quantique
_______________________________________________________________________________________________________________
### Auteur : Luc Carlos Asso

### Importation des Packages

In [ ]:
import numpy as np 
import dimod
from neal import SimulatedAnnealingSampler
import re
from docplex.mp.model import Model
from Instance import read_instance

### _______________________________________________________________________________________________________________

In [ ]:
class recuit_quantique :
    
    n, P, W, C, k = read_instance("Instance_190_100_100_10.txt")        #  A décocher après la prémière exécution 
    P = np.array(P)
    W = np.array(W)  
    sampler = SimulatedAnnealingSampler() 
    
    @staticmethod
    def QUBO ( λ : np.ndarray) :
        QUBO = {}
        for i in range(recuit_quantique.n) :
            if (i,i) in QUBO.keys(): 
                continue
            else :
                #QUBO [(i,i)] = -recuit_quantique.P[i,i] + λ[0] * recuit_quantique.W[i]  + λ[1]
                QUBO [(i,i)] = (-recuit_quantique.P[i,i] + λ[0] * (recuit_quantique.W[i]**2 - 2 * recuit_quantique.C * recuit_quantique.W[i])
                                           + λ[1] * (1 - 2*recuit_quantique.k))
            
            
            for j in range (i+1,recuit_quantique.n) :
                 QUBO[(i,j)] = -2*recuit_quantique.P[i,j] + 2 * λ[0] * (recuit_quantique.W[i]* recuit_quantique.W[j]) + 2 * λ[1]
        return QUBO

    @staticmethod
    def Moteur_Quantique(lam: np.ndarray):
        bqm = dimod.BinaryQuadraticModel.from_qubo(recuit_quantique.QUBO(lam))
    
        sampleset = recuit_quantique.sampler.sample(
            bqm,
            num_reads=30
        )
    
        best = sampleset.first.sample
        energy = sampleset.first.energy
    
        return best, energy

        
    @staticmethod
    def Constraints(solution):
        selected = [ i for i in range(recuit_quantique.n) if solution[i] == 1]
        weight = sum( recuit_quantique.W[i] for i in selected)
        cardinality = len(selected)
        return weight, cardinality,selected

    @staticmethod
    def RL_1(lambda_1, nombre_passages=20):
        n = recuit_quantique.n
        P = recuit_quantique.P
        W = recuit_quantique.W
        k = recuit_quantique.k
        score = np.diag(P) - lambda_1 * W
        selectionnes = set(np.argsort(score)[-k:])
        x = np.zeros(n, dtype=int)
        x[list(selectionnes)] = 1
        contribution = P @ x
    
        for _ in range(nombre_passages):
            meilleur_delta = 0
            meilleur_echange = None
            liste_selectionnes = list(selectionnes)
            liste_non_selectionnes = [j for j in range(n) if j not in selectionnes]
            
            for i in liste_selectionnes:
                for j in liste_non_selectionnes:
                    variation_profit = (
                        2 * (contribution[j] - contribution[i])
                        + P[i, i]
                        + P[j, j]
                        - 2 * P[i, j]
                    )
    
                    variation_penalite = -lambda_1 * (W[j] - W[i])
                    variation_totale = variation_profit + variation_penalite
    
                    if variation_totale > meilleur_delta:
                        meilleur_delta = variation_totale
                        meilleur_echange = (i, j)

            if meilleur_echange is None:
                break
            i, j = meilleur_echange
            x[i] = 0
            x[j] = 1
            selectionnes.remove(i)
            selectionnes.add(j)
            contribution = contribution - P[:, i] + P[:, j]
    
        return x

    @staticmethod
    def RL_2(lambda_2, nombre_passages=20):
        n = recuit_quantique.n
        P = recuit_quantique.P
        W = recuit_quantique.W
        C = recuit_quantique.C
        score = np.diag(P) - lambda_2
        ordre = np.argsort(score)[::-1]
        x = np.zeros(n, dtype=int)
        poids_total = 0
    
        for i in ordre:
            if score[i] > 0 and poids_total + W[i] <= C:
                x[i] = 1
                poids_total += W[i]
    
        selectionnes = set(np.where(x == 1)[0])
        contribution = P @ x
    
        for _ in range(nombre_passages):
            meilleure_variation = 0
            meilleure_operation = None
            liste_selectionnes = list(selectionnes)
            liste_non_selectionnes = [j for j in range(n) if j not in selectionnes]
    
            for j in liste_non_selectionnes:                                            #Tester les ajouts possibles.
                if poids_total + W[j] <= C:                                             # On ajoute un objet non sélectionné si la capacité le permet
                    variation_profit = 2 * contribution[j] + P[j, j]                           
                    variation_penalite = -lambda_2
                    variation_totale = variation_profit + variation_penalite
    
                    if variation_totale > meilleure_variation:
                        meilleure_variation = variation_totale
                        meilleure_operation = ("ajout", j)
    
            
            for i in liste_selectionnes:                                                 # Tester les retraits possibles.
                variation_profit = -2 * contribution[i] + P[i, i]                        # On retire un objet sélectionné si cela améliore l'objectif.
                variation_penalite = lambda_2
                variation_totale = variation_profit + variation_penalite
    
                if variation_totale > meilleure_variation:
                    meilleure_variation = variation_totale
                    meilleure_operation = ("retrait", i)
    

            for i in liste_selectionnes:
                for j in liste_non_selectionnes:
                    nouveau_poids = poids_total - W[i] + W[j]
    
                    if nouveau_poids <= C:
                        variation_profit = (
                            2 * (contribution[j] - contribution[i])
                            + P[i, i]
                            + P[j, j]
                            - 2 * P[i, j]
                        )
                        variation_penalite = 0
                        variation_totale = variation_profit + variation_penalite
    
                        if variation_totale > meilleure_variation:
                            meilleure_variation = variation_totale
                            meilleure_operation = ("echange", i, j)
    
        
            if meilleure_operation is None:                                              # On retire un objet sélectionné si cela améliore l'objectif.
                break
                
            if meilleure_operation[0] == "ajout":
                _, j = meilleure_operation
    
                x[j] = 1
                selectionnes.add(j)
                poids_total += W[j]
                contribution = contribution + P[:, j]
    
            elif meilleure_operation[0] == "retrait":
                _, i = meilleure_operation
                x[i] = 0
                selectionnes.remove(i)
                poids_total -= W[i]
                contribution = contribution - P[:, i]
    
            elif meilleure_operation[0] == "echange":
                _, i, j = meilleure_operation
                x[i] = 0
                x[j] = 1
                selectionnes.remove(i)
                selectionnes.add(j)
                poids_total = poids_total - W[i] + W[j]
                contribution = contribution - P[:, i] + P[:, j]
    
        return x

    @staticmethod
    def v_P():
        n = recuit_quantique.n
        mdl = Model("Primal")
        x = mdl.binary_var_list(n, name="x")
        obj = mdl.sum(recuit_quantique.P[i, i] * x[i] for i in range(n))
        obj += mdl.sum(
            2 * recuit_quantique.P[i, j] * x[i] * x[j]
            
            for i in range(n)
            for j in range(i + 1, n)
        )
    
        mdl.minimize(obj)
    
        mdl.add_constraint(mdl.sum(x) == recuit_quantique.k)
        mdl.add_constraint(
            mdl.sum(recuit_quantique.W[i] * x[i] for i in range(n)) <= recuit_quantique.C
        )
    
        mdl.parameters.timelimit = 30
        mdl.parameters.mip.tolerances.mipgap = 0.05
        mdl.parameters.threads = 0
        mdl.parameters.emphasis.mip = 1
    
        sol = mdl.solve(log_output=False)
    
        if sol is None:
            raise ValueError("Pas de solution trouvée")
    
        return np.array([int(sol[x[i]]) for i in range(n)])
        
    @staticmethod
    def Fonction_objectif( x : np.ndarray) :
        return x.T@ recuit_quantique.P @ x
    
    
    @staticmethod
    def Relaxation_Lagrangienne_λ_1(µ , d_k_1, λ_1_k, k , minornant_P) : 
        x = recuit_quantique.RL_1(λ_1_k)
        n = recuit_quantique.n
        d_k = recuit_quantique.C -np.dot(recuit_quantique.W, x)  + µ*d_k_1
        v_P = recuit_quantique.Fonction_objectif(minornant_P)
        v_RL_1_u_k = recuit_quantique.Fonction_objectif(x)
        eps = 1e-8
        u_k = (1/k)*( v_RL_1_u_k - v_P)/(abs(d_k) + eps)
        λ_1_k_1 =max(0.0 , λ_1_k - u_k*np.sign(d_k)) 
        return λ_1_k_1 , d_k

    @staticmethod
    def Relaxation_Lagrangienne_λ_2( µ , d_k_2, λ_2_k, k, minornant_P) : 
        x = recuit_quantique.RL_2(λ_2_k)
        n = recuit_quantique.n
        d_k = recuit_quantique.k - np.sum(x) + µ*d_k_2
        v_P = recuit_quantique.Fonction_objectif(minornant_P)
        v_RL_2_u_k = recuit_quantique.Fonction_objectif(x)
        eps = 1e-8
        u_k = (1/k)*(v_RL_2_u_k - v_P )/(abs(d_k) + eps)
        λ_2_k_1 =max(0.0 , λ_2_k - u_k*np.sign(d_k)) 
        return λ_2_k_1 , d_k

        
    @staticmethod
    def Relaxation_Lagrangienne(minornant_P , max_iter=100, µ = 2 ):
        lam = np.array([ 0,1])
        d_k =  np.array([1 , 1 ])
        best_solution = None
        best_energy = float("inf")
        for t in range(1,max_iter+1):
            solution, energy = (recuit_quantique.Moteur_Quantique(lam))
            weight, card,selected = (recuit_quantique.Constraints(solution))
            λ_1_k , λ_2_k = lam[0] , lam[1]
            d_k_1 , d_k_2 = d_k[0] , d_k[1]
            lam[0] , d_k[0] = recuit_quantique.Relaxation_Lagrangienne_λ_1( µ , d_k_1, λ_1_k, t, minornant_P)  # mise à jour des multiplicateurs
            lam[1] , d_k[1] = recuit_quantique.Relaxation_Lagrangienne_λ_2( µ , d_k_2, λ_2_k, t, minornant_P)  
            if energy < best_energy:                                                                           # sauvegarde meilleure solution
                best_energy = energy
                best_solution = solution
            print(f"Iteration {t}")
            print(f"Lambda = {lam}")
            print(f"Weight = {weight}")
            print(f"Cardinality = {card}")
            print(f"Energy = {energy}")
            print(selected)
            print("-"*40)
            _,card,selected = recuit_quantique.Constraints(best_solution)
        return selected,card, best_energy
  

In [ ]:
minornant_P = recuit_quantique.v_P()

In [ ]:
recuit_quantique.Relaxation_Lagrangienne(minornant_P)

### reconstitution de la solution 

In [ ]:
def recons_SOL( ind : list ):
    x= [0]*recuit_quantique.n
    for i in ind : 
        x[i] = 1
    return np.array(x)

### Valeur de la solution par Fonction_objectif

In [ ]:
x =recons_SOL([11, 28, 34, 49, 50, 68, 75, 84, 102, 106, 120, 121, 156, 158, 171] )
recuit_quantique.Fonction_objectif(x)

In [ ]:
x